# CPG-RL 訓練：智元 D1 EDU **輪足版（ZSL-1w）** · MJX · Colab GPU

移植自 task4 的 Go2 論文標準版。四顆輪子在模擬中熔接鎖死，當成 71 mm 圓腳走路，
因此動作空間仍是 12 維。與 Go2 版的差異：

| 項目 | Go2（task4）| D1 輪足（本檔）|
|---|---|---|
| home 關節角 | `[0, 0.9, -1.8]` | `[0, 1.05, -2.00]` |
| PD | `apply_pd()` 覆寫 90/3 | XML 內建 kp=80/kd=1（原廠 demo 值），**無 apply_pd** |
| 力矩上限 | 23.7 / knee 45.43 | 28 全關節 |
| 負重 DR | 0~8 kg | 0~5 kg（官方額定 payload）|
| 總質量 | 15 kg | **20.56 kg**（四顆輪各 0.9 kg）|
| 步幅尺度 | 前後/側向同為 0.12 | 前後 0.12、**側向 0.09**（abad 行程僅 ±28°）|
| obs | 76 維 | **69 維** = 76 −3（機身線速度，實機 LowLevel 拿不到）−4（腳觸地布林）|
| 觸地判定 | 腳掌世界高度 < 3 cm | **移除**（實測無可用訊號，見關卡 3）|
| 高度獎勵基準 | keyframe z | **0.2695**（實際站定高度，非 keyframe 的 0.2948）|
| IMU DR | 無 | 重力/角速度加雜訊與偏差 |

執行階段 → 變更類型 → GPU。先跑 Smoke test 再開訓練。

---

## ⚠️ 常數是刻意的重複——改一邊就要改另一邊

Colab 無法 import 本地模組，所以下面 Cell 5 的常數是
`task6/inference/d1_model.py` 的**手抄副本**。兩份常數一旦分岔，
訓練出來的權重在本機推論時會靜默走樣（維度對得上、行為對不上，不會有任何錯誤訊息）。

**任何一邊改了常數，另一邊必須同步改。** 涉及的常數：

`MU_MIN` `MU_MAX` `OMEGA_MIN` `OMEGA_MAX` `A_CONV` `D_STEP` `D_STEP_Y`
`G_C` `G_P` `NOMINAL_HEIGHT` `W_COUP` `N_CPG_SUB` `CTRL_DT` `SIM_DT`
`KP` `KD` `TAU_MAX` `OBS_DIM` `ACT_DIM` `HOME3` `LEGS` `PHASE_OFFSET`

（本檔的 `KP_NOM` / `KD_NOM` / `HOME3_np` 對應 d1_model 的 `KP` / `KD` / `HOME3`。）

`_obs` 的**欄位順序**同樣是重複：必須與 `task6/inference/obs_d1.py` 的
`OBS_LAYOUT` 逐項一致，否則權重同樣會靜默失效。


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
!pip install -q mujoco mujoco-mjx brax mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

In [ ]:
import os, subprocess

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "feat/d1-edu-cpg-rl"    # task6/ 只在這個功能分支；併進 main 後改成 "main"
DEST = "rbtdog_sim"              # 明寫目的地：repo 名是大寫 RBTDOG_SIM，預設會 clone 成別的資料夾

if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST],
                   check=True)
SCENE = f"{DEST}/task6/model/d1_edu_w/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

# 抓不到就當場停：Cell 6 才炸的話錯誤訊息會指向 MuJoCo，看不出真正原因
assert os.path.exists(SCENE), (
    f"抓不到 {SCENE}。\n"
    f"1) 確認分支 {BRANCH} 已 push 上 {REPO}（本機分支沒 push 的話 clone 不到）；\n"
    f"2) 或改用左側檔案面板，把整個 task6/model/d1_edu_w/（含 meshes/ 的 17 個 STL）"
    f"上傳到 {DEST}/task6/model/d1_edu_w/。"
)

In [ ]:
import jax.numpy as jnp
import numpy as np

MU_MIN, MU_MAX = 1.0, 2.0
OMEGA_MIN, OMEGA_MAX = 0.0, 4.5
A_CONV = 50.0
D_STEP, D_STEP_Y = 0.12, 0.09    # 前後 / 側向；側向較小是因 abad 行程僅 ±28°
G_C, G_P = 0.08, 0.01
NOMINAL_HEIGHT = 0.2695          # 實際站定高度（keyframe 的 0.2948 是純運動學值）
W_COUP = 8.0
N_CPG_SUB = 4
CTRL_DT, SIM_DT = 0.02, 0.004
KP_NOM, KD_NOM = 80.0, 1.0          # 原廠 demo 值
TAU_MAX = 28.0
OBS_DIM, ACT_DIM = 69, 12
HOME3_np = np.array([0.0, 1.05, -2.00])
HOME12 = jnp.array(list(HOME3_np) * 4)
LEGS = ["FL", "FR", "RL", "RR"]

PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])
PHI = PHASE_OFFSET[None, :] - PHASE_OFFSET[:, None]


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4),
            "theta": PHASE_OFFSET}


def cpg_step(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI
        coup = jnp.sum(rbar[None, :] * jnp.sin(diff), axis=1)
        th = th + (2.0 * jnp.pi * omega + W_COUP * coup) * h
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd,
            "theta": jnp.mod(th, 2.0 * jnp.pi)}


def action_to_cpg_cmd(action):
    a = jnp.tanh(action).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    om = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, om

In [ ]:
import mujoco

def leg_ik_consts(xml):
    m = mujoco.MjModel.from_xml_path(xml); d = mujoco.MjData(m)
    f0s, jinvs = [], []
    for k, leg in enumerate(LEGS):
        jb = 7 + 3 * k
        gid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg)
        hip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, leg + "_abad")
        def foot(q3):
            mujoco.mj_resetDataKeyframe(m, d, 0)
            d.qpos[jb:jb + 3] = q3; mujoco.mj_forward(m, d)
            return (d.geom_xpos[gid] - d.xpos[hip]).copy()
        f0 = foot(HOME3_np); e = 1e-3; J = np.zeros((3, 3))
        for j in range(3):
            dq = np.zeros(3); dq[j] = e
            J[:, j] = (foot(HOME3_np + dq) - foot(HOME3_np - dq)) / (2 * e)
        f0s.append(f0); jinvs.append(np.linalg.inv(J))
    return np.array(f0s, np.float32), np.array(jinvs, np.float32)

F0S_np, JINVS_np = leg_ik_consts(SCENE)
print("f0 每腿(x,y,z 相對髖):\n", np.round(F0S_np, 3))

In [ ]:
import functools
from brax.envs.base import Env, State
from mujoco import mjx

N_FRAMES = int(round(CTRL_DT / SIM_DT))
PUSH_EVERY = 100          # 每 100 控制步(=2s) 注入一次速度擾動
PUSH_VEL = 0.6
GRAV_NOISE = 0.02         # 重力向量雜訊 sigma（IMU 為原始資料、精度一般）
GYRO_NOISE = 0.10         # 角速度雜訊 sigma (rad/s)
GYRO_BIAS = 0.05          # 每 episode 取樣一次的角速度偏差上限 (rad/s)


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)
def w2b(quat, v): return _qrot(_qinv(quat), v)


class D1wCpgEnv(Env):
    def __init__(self, f0s, jinvs):
        m = mujoco.MjModel.from_xml_path(SCENE)
        m.opt.timestep = SIM_DT
        self._mj = m
        self.sys = mjx.put_model(m)
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._f0s = jnp.array(f0s)
        self._jinvs = jnp.array(jinvs)

    # ---- 動作 → 關節目標角 ----
    def _joint_targets(self, c):
        th = c["theta"]
        fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
        fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
        dx = -D_STEP * fx * jnp.cos(th)
        dy = D_STEP_Y * fy * jnp.cos(th)
        dz = jnp.where(jnp.sin(th) > 0, G_C * jnp.sin(th), G_P * jnp.sin(th))
        off = jnp.stack([dx, dy, dz], -1)
        q = jax.vmap(lambda ji, o: jnp.array(HOME3_np) + ji @ o)(self._jinvs, off)
        return q.reshape(12)

    # ---- 69 維 obs（欄位順序必須與 task6/inference/obs_d1.py 一致）----
    def _obs(self, data, c, cmd, last_a, gyro_bias):
        quat = data.qpos[3:7]
        grav = w2b(quat, jnp.array([0.0, 0.0, -1.0]))
        gyro = data.qvel[3:6] + gyro_bias
        return jnp.concatenate([
            grav, gyro,
            data.qpos[7:19] - HOME12, data.qvel[6:18],
            cmd, last_a,
            c["rx"], c["rx_d"], c["ry"], c["ry_d"],
            jnp.sin(c["theta"]), jnp.cos(c["theta"]),
        ])

    def reset(self, rng):
        k_cmd, k_bias, k_noise = jax.random.split(rng, 3)
        data = mjx.make_data(self.sys).replace(qpos=self._init_q,
                                               ctrl=jnp.array(self._mj.key_ctrl[0]))
        data = mjx.forward(self.sys, data)
        cmd = jnp.array([jax.random.uniform(k_cmd, minval=0.3, maxval=1.0), 0.0, 0.0])
        gyro_bias = jax.random.uniform(k_bias, (3,), minval=-GYRO_BIAS, maxval=GYRO_BIAS)
        c = cpg_init()
        info = {"rng": k_noise, "c": c, "last_a": jnp.zeros(ACT_DIM),
                "cmd": cmd, "gyro_bias": gyro_bias, "step": 0}
        obs = self._obs(data, c, cmd, jnp.zeros(ACT_DIM), gyro_bias)
        metrics = {"height": data.qpos[2], "vx": 0.0}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        info = dict(state.info)
        mux, muy, om = action_to_cpg_cmd(action)
        c = cpg_step(info["c"], mux, muy, om, CTRL_DT)
        q_des = jnp.clip(self._joint_targets(c), self._lo, self._hi)

        data = state.pipeline_state
        rng, k_push, k_dir, k_obs = jax.random.split(info["rng"], 4)
        do_push = (info["step"] % PUSH_EVERY) == (PUSH_EVERY - 1)
        ang = jax.random.uniform(k_dir, minval=0.0, maxval=2 * jnp.pi)
        mag = jax.random.uniform(k_push, minval=0.0, maxval=PUSH_VEL)
        kick = jnp.where(do_push,
                         jnp.array([mag * jnp.cos(ang), mag * jnp.sin(ang), 0.0]),
                         jnp.zeros(3))
        data = data.replace(qvel=data.qvel.at[0:3].add(kick))

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=q_des)), None
        data, _ = jax.lax.scan(one, data, None, length=N_FRAMES)

        grav = w2b(data.qpos[3:7], jnp.array([0.0, 0.0, -1.0]))
        vb = w2b(data.qpos[3:7], data.qvel[0:3])      # reward 用真值速度（只在模擬）
        cmd = info["cmd"]

        r_vx = jnp.exp(-4.0 * (vb[0] - cmd[0]) ** 2)
        r_vy = jnp.exp(-4.0 * vb[1] ** 2)
        r_wz = jnp.exp(-4.0 * (data.qvel[5] - cmd[2]) ** 2)
        r_h = jnp.exp(-40.0 * (data.qpos[2] - NOMINAL_HEIGHT) ** 2)   # 用實際站定高度，非 keyframe
        c_act = jnp.sum((action - info["last_a"]) ** 2)
        c_tau = jnp.sum(data.actuator_force ** 2)
        reward = 1.5 * r_vx + 0.5 * r_vy + 0.5 * r_wz + 0.5 * r_h \
                 - 0.01 * c_act - 2e-4 * c_tau

        done = jnp.where(grav[2] > -0.4, 1.0, 0.0)

        noise = jax.random.normal(k_obs, (6,))
        obs = self._obs(data, c, cmd, action, info["gyro_bias"])
        obs = obs.at[0:3].add(GRAV_NOISE * noise[0:3])
        obs = obs.at[3:6].add(GYRO_NOISE * noise[3:6])

        info.update({"rng": rng, "c": c, "last_a": action, "step": info["step"] + 1})
        metrics = {"height": data.qpos[2], "vx": vb[0]}
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)

    @property
    def observation_size(self): return OBS_DIM

    @property
    def action_size(self): return ACT_DIM

    @property
    def backend(self): return "mjx"

In [ ]:
_mm = mujoco.MjModel.from_xml_path(SCENE)
BASE_ID = mujoco.mj_name2id(_mm, mujoco.mjtObj.mjOBJ_BODY, "base")

def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5 = jax.random.split(rng, 5)
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.3, maxval=1.0))
        kp = jax.random.uniform(k2, minval=60.0, maxval=100.0)    # 名目 80
        kd = jax.random.uniform(k3, minval=0.5, maxval=2.0)       # 名目 1
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.8, maxval=1.2)             # 連桿質量 ±20%
        payload = jax.random.uniform(k5, minval=0.0, maxval=5.0)  # 官方額定 payload 5 kg
        body_mass = body_mass.at[BASE_ID].add(payload)
        return geom_friction, gain, bias, body_mass
    gf, gain, bias, bm = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm)
    return sys, in_axes
print("domain_randomize ready")

In [ ]:
env = D1wCpgEnv(F0S_np, JINVS_np)

# MJX 上踩過的坑（task4 地形版）：actuator 的 biastype 若不是 affine，
# ctrl 會被當成力矩直接施加，機器人會直接塌掉。XML 已宣告 affine，
# 這裡確認 mjx.put_model 之後仍然保持。
assert env.sys.actuator_biastype[0] == mujoco.mjtBias.mjBIAS_AFFINE, \
    "actuator biastype 不是 affine，ctrl 會被當力矩施加 → 機器人會塌掉"

s = jax.jit(env.reset)(jax.random.PRNGKey(0))
print("obs shape:", s.obs.shape, "(應為 (69,))")
s = jax.jit(env.step)(s, jnp.zeros(12))
print("reward:", float(s.reward), "done:", float(s.done), "height:", float(s.metrics["height"]))
assert s.obs.shape == (69,), "obs 維度不對，回頭對 _obs 的欄位順序"
assert np.isfinite(float(s.reward)), "reward 出現 NaN/Inf"
print("PASSED")

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = D1wCpgEnv(F0S_np, JINVS_np)
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

train_fn = functools.partial(
    ppo.train, num_timesteps=120_000_000, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=domain_randomize, seed=0)

_t0 = time.time(); rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    print(f"step {step:>10,}  reward {r:8.2f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()

In [ ]:
from brax.io import model
model.save_params("cpg_rl_d1w_params.pkl", params)
try:
    from google.colab import files; files.download("cpg_rl_d1w_params.pkl")
except Exception as e:
    print("左側檔案面板右鍵下載 cpg_rl_d1w_params.pkl。", e)